# Automatic Differentiation (Gradient Calculation) of Tensors in PyTorch

In [1]:
import torch

## 1. Simple function differentiation

- d(x^2)/dx = 2x

In [2]:
x = torch.tensor(3.0, requires_grad=True)
x

tensor(3., requires_grad=True)

In [3]:
y = x**2
y

tensor(9., grad_fn=<PowBackward0>)

In [4]:
y.backward()

In [5]:
x.grad

tensor(6.)

## 2. Nested function differentiation

- dz/dx where z = sin(y) and, y = x^2 

In [10]:
x = torch.tensor(3.0, requires_grad=True)
x

tensor(3., requires_grad=True)

In [11]:
y = x**2
y

tensor(9., grad_fn=<PowBackward0>)

In [12]:
z = torch.sin(y)
z

tensor(0.4121, grad_fn=<SinBackward0>)

In [13]:
z.backward()

In [14]:
x.grad

tensor(-5.4668)

## 3. Manual Simulation of a Complete Epoch for a Single Perceptron/Neuron

In [16]:
# Inputs
x = torch.tensor(6.7) # Input feature
y = torch.tensor(0.0) # True label (binary)

w = torch.tensor(1.0) # Weight
b = torch.tensor(0.0) # Bias

In [18]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8 # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [20]:
# Forward pass
z = w*x + b # Weighted sum (linear part)
y_pred = torch.sigmoid(z) # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [23]:
# Derivatives (Backpropagation)

# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x # dz/dw = x
dz_db = 1 # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [24]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


## 4. Simulation of a Complete Epoch for a Single Perceptron/Neuron using PyTorch

In [26]:
# Inputs
x = torch.tensor(6.7) # Input feature
y = torch.tensor(0.0) # True label (binary)

In [28]:
w = torch.tensor(1.0, requires_grad=True) # Weight
b = torch.tensor(0.0, requires_grad=True) # Bias

In [29]:
# Forward pass
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [30]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [31]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [32]:
# backpropagation
loss.backward()

In [33]:
print(f"Gradient of loss w.r.t weight (dw): {w.grad}")
print(f"Gradient of loss w.r.t bias (db): {b.grad}")

Gradient of loss w.r.t weight (dw): 6.6917619705200195
Gradient of loss w.r.t bias (db): 0.9987704753875732


## 5. Gradient Calculation on Vectors

In [34]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
x

tensor([1., 2., 3.], requires_grad=True)

In [36]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [37]:
y.backward()

In [38]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

## 6. Clearing Gradients

In [64]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [65]:
y  = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [66]:
y.backward(retain_graph=True)
# y.backward()

In [67]:
x.grad

tensor(4.)

In [68]:
y.backward(retain_graph=True)

In [69]:
# Repeating the backpropagation accumulates gradients
x.grad

tensor(8.)

- Use x.grad.zero_() before each backward call to clear any previous gradients

In [70]:
x.grad.zero_()
y.backward(retain_graph=True)

In [71]:
x.grad

tensor(4.)

## 7. Disable Gradient Tracking

In [72]:
# Disable gradient tracking for inference stage
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [73]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [74]:
y.backward()

In [75]:
x.grad

tensor(4.)

### a. requires_grad_(False)

In [76]:
x.requires_grad_(False)

tensor(2.)

In [78]:
y = x**2
y

tensor(4.)

### b. detach()

In [79]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [82]:
z = x.detach()
z, x

(tensor(2.), tensor(2., requires_grad=True))

### c. torch.no_grad()

In [83]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [84]:
with torch.no_grad():
    y = x ** 2

In [85]:
y

tensor(4.)

In [86]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)